# Academic Citation Network Data Processing

This notebook transforms the AMiner citation dataset into structured CSV files for Neo4j.

The workflow extracts article, author, research-field, authorship, citation, and article-field relationships. It is designed to be run from the `data-processing/` directory with the project's `.venv` kernel.

> The full AMiner dataset is very large. Test the notebook with a smaller sample before processing the complete dataset.

In [14]:
from pathlib import Path
import ast
import glob
import ijson
import os
import re
from decimal import Decimal

import pandas as pd


NOTEBOOK_DIR = Path.cwd()
PROJECT_ROOT = NOTEBOOK_DIR.parent
DATA_DIR = PROJECT_ROOT / "data"
INPUT_JSON = DATA_DIR / "dblp_v14.json"
OUTPUT_DIR = DATA_DIR
BATCH_SIZE = 100_000
CHUNK_SIZE = 250_000

In [15]:
def none_if_empty(value):
    """Return None for empty values while preserving meaningful data."""
    return None if value == "" else value

In [16]:
def save_batch(records, batch_number):
    """Save one streamed batch as an intermediate CSV file."""
    batch_path = OUTPUT_DIR / f"partial_batch_{batch_number}.csv"
    pd.DataFrame(records).to_csv(batch_path, index=False)
    return batch_path

In [17]:
records = []
batch_number = 0

with INPUT_JSON.open("r", encoding="utf-8") as source_file:
    for index, paper in enumerate(ijson.items(source_file, "item"), start=1):
        authors = paper.get("authors", [])
        fields_of_study = paper.get("fos", [])

        records.append({
            "id": paper.get("id"),
            "title": paper.get("title"),
            "authorsName": [none_if_empty(author.get("name")) for author in authors],
            "authorsOrganization": [none_if_empty(author.get("org")) for author in authors],
            "authorsId": [none_if_empty(author.get("id")) for author in authors],
            "venue": paper.get("venue", {}).get("raw", ""),
            "year": paper.get("year"),
            "keywords": paper.get("keywords", []),
            "fieldOfStudy": [field.get("name") for field in fields_of_study],
            "weightOfFieldOfStudy": [field.get("w") for field in fields_of_study],
            "references": paper.get("references", []),
            "n_citation": paper.get("n_citation"),
            "page_start": paper.get("page_start"),
            "page_end": paper.get("page_end"),
            "doc_type": paper.get("doc_type"),
            "lang": paper.get("lang"),
            "volume": paper.get("volume"),
            "issue": paper.get("issue"),
            "issn": paper.get("issn"),
            "isbn": paper.get("isbn"),
            "doi": paper.get("doi"),
            "url": ", ".join(paper.get("url", [])),
            "abstract": paper.get("abstract"),
        })

        if len(records) >= BATCH_SIZE:
            save_batch(records, batch_number)
            records = []
            batch_number += 1

if records:
    save_batch(records, batch_number)

print(f"Streamed {index:,} papers into {batch_number + (1 if records else 0)} batch files.")

Streamed 5,259,858 papers into 53 batch files.


In [ ]:
batch_files = sorted(DATA_DIR.glob("partial_batch_*.csv"))
if not batch_files:
    raise FileNotFoundError(f"No batch files found in {DATA_DIR}")

df_complete = pd.concat(
    (pd.read_csv(batch_file) for batch_file in batch_files),
    ignore_index=True,
)

print(f"Reconstructed dataframe: {len(df_complete):,} rows and {len(df_complete.columns)} columns")

In [ ]:
complete_csv = DATA_DIR / "complete.csv"
df_complete.to_csv(complete_csv, index=False)
print(f"Saved intermediate dataset to {complete_csv}")

In [20]:
df = pd.read_csv(complete_csv)
print(f"Loaded {len(df):,} rows from {complete_csv.name}")

Loaded 5,259,858 rows from complete.csv


In [ ]:
LIST_COLUMNS = ["references", "authorsName", "authorsOrganization", "authorsId", "fieldOfStudy"]
WEIGHT_COLUMN = "weightOfFieldOfStudy"


def parse_list(value):
    """Convert a serialized Python list into a list without raising on bad values."""
    if pd.isna(value) or str(value).strip() == "":
        return []
    try:
        parsed = ast.literal_eval(value)
        return parsed if isinstance(parsed, list) else []
    except (SyntaxError, ValueError, TypeError):
        return []


def parse_weights(value):
    """Convert serialized Decimal values into floating-point weights."""
    parsed = parse_list(value)
    return [float(item) for item in parsed if item is not None]


df_final = pd.read_csv(complete_csv)
for column in LIST_COLUMNS:
    df_final[column] = df_final[column].map(parse_list)
df_final[WEIGHT_COLUMN] = df_final[WEIGHT_COLUMN].map(parse_weights)

print(f"Parsed {len(df_final):,} records")

## Data Cleaning

Normalize serialized list columns and remove escaped characters before preparing the graph tables.

In [ ]:
df = df.applymap(lambda x: re.sub(r'\\"', '"', x) if isinstance(x, str) else x)

## Citation Graph Preparation

Build an undirected neighbour representation to identify connected article components.

Select the article identifiers and their reference lists.

In [ ]:
df_final[["fieldOfStudy", "weightOfFieldOfStudy"]].iloc[20:200]

,fieldOfStudy,weightOfFieldOfStudy
20,[],[]
21,[],[]
22,[],[]
23,[],[]
24,[],[]
...,...,...
195,"[Medical education, Curriculum development, Ma...","[0.47043, 0.60328, 0.49063, 0.40307, 0.0, 0.45..."
196,"[Program slicing, Graph, Programming language,...","[0.76926, 0.0, 0.46314, 0.45716, 0.51093, 0.46..."
197,"[Software engineering, Computer science, Agile...","[0.4692, 0.46982, 0.52139, 0.46147, 0.56207, 0..."
198,"[Discrete mathematics, Concatenated error corr...","[0.42223, 0.5785, 0.64297, 0.54471, 0.4123, 0...."


In [ ]:
df_cc = df_final[["id", "references"]].copy()

Expand each reference list so every citation becomes one row.

In [ ]:
df_cc = df_cc.explode("references").reset_index(drop=True)

In [ ]:
df_cc.dropna(inplace=True)

Create a neighbour list for every article.

In [ ]:
df_cc_2 = df_cc.copy()

In [ ]:
df_cc_2.columns = ["references", "id"]

In [ ]:
df_cc = pd.concat([df_cc, df_cc_2], ignore_index=True)

In [ ]:
df_cc.drop_duplicates(inplace=True)

In [ ]:
df_cc = df_cc.groupby('id')['references'].agg(lambda x: sorted(set(x))).reset_index()

In [ ]:
df_cc.columns = ["id", "neighbors"]

In [ ]:
del df_cc_2

Sort articles by the number of connected neighbours.

In [ ]:
df_cc["neighbors_count"] = df_cc["neighbors"].str.len()
df_cc["component"] = None
df_cc = df_cc.set_index("id").sort_values("neighbors_count", ascending=False)

In [ ]:
df_cc = df_cc.sort_values(by='neighbors_count', ascending=False)

In [ ]:
df_cc.head(50)

,neighbors,neighbors_count,Component
id,,,
53e9986eb7602d97020ab93b,"[53e99792b7602d9701f58212, 53e9979bb7602d9701f...",10940,None
573696026e3b12023e515eec,"[53e99a85b7602d97022f8644, 53e99c7cb7602d97025...",10190,None
53e9a95db7602d97032b5715,"[53e9979eb7602d9701f6b20e, 53e9979eb7602d9701f...",9731,None
53e9b61bb7602d97041735d8,"[53e9979bb7602d9701f63a64, 53e997d7b7602d9701f...",8858,None
53e9b48fb7602d9703f998f7,"[53e997b2b7602d9701f94fb1, 53e997e3b7602d9701f...",8797,None
53e9bcc1b7602d97049412d4,"[53e997b5b7602d9701f9ba97, 53e997bab7602d9701f...",8564,None
53e99a85b7602d97022f8644,"[53e99792b7602d9701f581d2, 53e997b5b7602d9701f...",8076,None
53e9a281b7602d9702b88a98,"[53e997c6b7602d9701fb7074, 53e997f1b7602d9701f...",8051,None
53e9b47cb7602d9703f7ee0d,"[53e99792b7602d9701f544ea, 53e997a2b7602d9701f...",7599,None


In [ ]:
component_count = 0
unassigned_nodes = df_cc.loc[df_cc["component"].isna()]

while not unassigned_nodes.empty:
    node_id = unassigned_nodes.index[0]
    df_cc.loc[node_id, "component"] = component_count
    frontier = [node_id]

    while frontier:
        neighbours = df_cc.loc[frontier, "neighbors"].explode().dropna().tolist()
        neighbours = list(set(neighbours) & set(df_cc.index))
        unassigned_neighbours = df_cc.loc[neighbours]
        unassigned_neighbours = unassigned_neighbours[
            unassigned_neighbours["component"].isna()
        ]
        frontier = unassigned_neighbours.index.tolist()[:100]
        df_cc.loc[frontier, "component"] = component_count

    component_count += 1
    unassigned_nodes = df_cc.loc[df_cc["component"].isna()]
    break

print(f"Identified {component_count} connected component(s) in the current sample.")


New Component: 0
10940
(100,)
Total nodos recorridos: 101
(100,)
Total nodos recorridos: 201
(100,)
Total nodos recorridos: 301
(100,)
Total nodos recorridos: 401
(100,)
Total nodos recorridos: 501
(100,)
Total nodos recorridos: 601
(100,)
Total nodos recorridos: 701
(100,)
Total nodos recorridos: 801
(100,)
Total nodos recorridos: 901
(100,)
Total nodos recorridos: 1001


In [ ]:
df_cc

,neighbors,neighbors_count,Component
id,,,
53e9986eb7602d97020ab93b,"[53e99792b7602d9701f58212, 53e9979bb7602d9701f...",10940,0
573696026e3b12023e515eec,"[53e99a85b7602d97022f8644, 53e99c7cb7602d97025...",10190,None
53e9a95db7602d97032b5715,"[53e9979eb7602d9701f6b20e, 53e9979eb7602d9701f...",9731,None
53e9b61bb7602d97041735d8,"[53e9979bb7602d9701f63a64, 53e997d7b7602d9701f...",8858,None
53e9b48fb7602d9703f998f7,"[53e997b2b7602d9701f94fb1, 53e997e3b7602d9701f...",8797,None
...,...,...,...
53e9b666b7602d97041ca9f7,[53e9985fb7602d9702097504],1,None
53e9b666b7602d97041caa36,[53e9bd76b7602d9704a19a3a],1,None
53e9b666b7602d97041caa57,[53e9bc5bb7602d97048d49eb],1,None


In [ ]:
df_final = df_final.set_index("id")

Select the first connected component and remove references to missing articles.

In [ ]:
missing_reference_ids = [
    "557d0405f667eeed5619645d",
    "558abe4084ae84d265bf60a2",
    "558a9e9be4b031bae1f8b0a5",
    "5583e06a0cf2a1f3dc49c9ad",
    "619b55521c45e57ce9b8d8d8",
    "557e4b14f6678c77ea221d26",
    "5ac1829d17c44a1fda917d40",
    "599c7cbf601a182cd27d28a4",
]

df_cc = df_cc.loc[~df_cc.index.isin(missing_reference_ids)]
component_zero_ids = df_cc.loc[df_cc["component"] == 0].index
df = df_final.loc[component_zero_ids].reset_index()

## Article Nodes

In [ ]:
lista_campos_articulos = ["id", "title", "venue", "year", "keywords", "n_citation", "page_start", "page_end", "doc_type", "lang",
                "volume", "issue", "issn", "isbn", "doi", "url", "abstract"]
lista_campos_NEO4j = ["id", "title", "year", "url"]

In [ ]:
df_articles = df[lista_campos_NEO4j].copy()

## Article-Citation Relationships

In [ ]:
df_articles_articles = df[["id", "references"]].copy()
df_articles_articles = df_articles_articles.explode("references").reset_index(drop=True)
df_articles_articles

,id,references
0,53e9986eb7602d97020ab93b,558a2b07e4b037c087558e13
1,53e9986eb7602d97020ab93b,5c78ad854895d9cbc6d9ebb0
2,53e9986eb7602d97020ab93b,53e99808b7602d970201b87e
3,53e9986eb7602d97020ab93b,53e99822b7602d970204287c
4,53e9986eb7602d97020ab93b,53e99b0ab7602d970239b0e3
...,...,...
21911,53e9b9c1b7602d97045b6d09,557d1ddd6feeaa8086da6bd9
21912,5a9cb63417c44a376ffb60d1,NaN
21913,53e9b61bb7602d9704170608,53e9a0d1b7602d97029b7a9e
21914,573698286e3b12023e6fa2d0,53e9b587b7602d97040c7931


Inspect the relationship table before cleaning.

In [ ]:
df_articles_articles.columns = ["article_citant", "article_citat"]

Remove rows with missing identifiers.

In [ ]:
df_articles_articles.dropna(inplace=True)

Remove duplicate citation relationships.

In [ ]:
df_articles_articles.drop_duplicates(inplace=True)

In [ ]:
df_articles_articles.shape

(19568, 2)

## Article-Author Relationships

In [ ]:
lista_campos_prepare_articles = ["id", 'name', 'organization', 'authorsId']
df_prepare_articles = df[lista_campos_prepare_articles]

In [ ]:
df_prepare_articles = df_prepare_articles.explode(["name", "organization", "authorsId"]).reset_index(drop=True)

Create the article-author node and relationship tables.

In [ ]:
df_articles_autors = df_prepare_articles[["id", "authorsId"]].copy()
df_autors          = df_prepare_articles[["authorsId", "name", "organization"]].copy()

Normalize escaped characters in author data.

In [ ]:
df_autors = df_autors.applymap(lambda x: re.sub(r'\\"', '"', x) if isinstance(x, str) else x)

Remove authors and relationships with missing identifiers.

In [ ]:
df_autors.dropna(subset=["authorsId"], inplace=True)
df_articles_autors.dropna(inplace=True)

Remove duplicate authors and article-author relationships.

In [ ]:
df_articles_autors.drop_duplicates(inplace=True)
df_autors.drop_duplicates(subset=["authorsId"], inplace=True)

## Article-Research Field Relationships

In [ ]:
df_prepare_fos = df[["id", "fieldOfStudy", "weightOfFieldOfStudy"]]
df_prepare_fos = df_prepare_fos.explode(["fieldOfStudy", "weightOfFieldOfStudy"]).reset_index(drop=True)

Create the article-research-field and research-field node tables.

In [ ]:
df_articles_fos = df_prepare_fos[["id", "fieldOfStudy", "weightOfFieldOfStudy"]].copy()
df_fos          = df_prepare_fos[["fieldOfStudy"]].copy()

Remove rows with missing research-field values.

In [ ]:
df_fos.dropna(subset=["fieldOfStudy"], inplace=True)
df_articles_fos.dropna(inplace=True)

Remove duplicate research-field nodes and relationships.

In [ ]:
df_fos.drop_duplicates(inplace=True)
df_articles_fos.drop_duplicates(inplace=True)

## Export Neo4j Tables

In [ ]:
df_articles = df_articles.applymap(lambda x: re.sub(r'\\"', '"', x) if isinstance(x, str) else x)

In [ ]:
saving_directory = OUTPUT_DIR
saving_directory.mkdir(parents=True, exist_ok=True)
print(f"CSV output directory: {saving_directory}")

In [ ]:
df_articles.to_csv         (os.path.join(saving_directory, "articles.csv"),              index=False)
df_autors.to_csv           (os.path.join(saving_directory, "autors.csv"),                index=False)
df_fos.to_csv              (os.path.join(saving_directory, "fieldOfStudy.csv"),          index=False)
df_articles_autors.to_csv  (os.path.join(saving_directory, "articles_autors.csv"),       index=False)
df_articles_articles.to_csv(os.path.join(saving_directory, "articles_articles.csv"),     index=False)
df_articles_fos.to_csv     (os.path.join(saving_directory, "articles_fieldOfStudy.csv"), index=False)